# Notebook 1: The time series inventory

What does the system hold? This notebook shows the inventory of the retrieval server as a table, one chosen series
as a curve with its coverage per day and its gaps, and the electricity price of 2025 with its change of resolution.

**Rules of this notebook.** It only reads; it writes nothing to the broker and nothing to the system. Gaps stay
gaps. Every table carries the origin of a series: measured, reported, estimated, simulated.

Two paths to the data: **A** through the retrieval server (the normal case), **B** directly from the topic through
the broker, when A cannot be reached. The notebook chooses itself and says which path it takes.

This is a copy that is renewed when the notebook server starts. To keep working with it, save it under a name of
your own.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from opentsx.client import RetrievalClient, TopicReader
from opentsx.model.pandas_io import episodes_to_frame

RETRIEVAL = os.environ.get("OPENTSX_RETRIEVAL", "http://opentsx-retrieval.opentsx.svc:8080")
BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP")
DOMAIN = os.environ.get("OPENTSX_DOMAIN", "wind")
TOPIC = f"tsx.{DOMAIN}.episodes.v2"


def reachable(url: str) -> bool:
    try:
        with urllib.request.urlopen(url + "/v1/health", timeout=5) as r:
            return r.status == 200
    except OSError:
        return False


WAY = "A" if reachable(RETRIEVAL) else ("B" if BOOTSTRAP else None)
print({"A": f"Path A: retrieval server {RETRIEVAL}",
       "B": f"Path B: directly from {TOPIC} through {BOOTSTRAP} (the retrieval server cannot be reached)",
       None: "Neither the retrieval server nor a broker can be reached: set OPENTSX_RETRIEVAL or KAFKA_BOOTSTRAP."}[WAY])
client = RetrievalClient(RETRIEVAL)
reader = TopicReader(BOOTSTRAP) if BOOTSTRAP else None

## 1. The inventory

One row per series: what it measures, in which unit, from when to when, how many values, how many are missing, and
where it comes from. On path B the table is built from the episodes of the topic; that takes longer and loads the
broker, so the reader reads at most as many records as `MAX_RECORDS` allows.

In [ ]:
MAX_RECORDS = 20_000


def inventory_from_topic() -> pd.DataFrame:
    eps = reader.episodes(TOPIC, max_records=MAX_RECORDS)
    rows = {}
    for e in eps:
        s = e["series"]
        r = rows.setdefault(s["seriesId"], {"series_id": s["seriesId"], "metric": s["metric"], "unit": s.get("unit"),
                                            "from": e["tStart"], "to": e["tEnd"], "samples": 0,
                                            "origin": (e.get("labels") or {}).get("origin"),
                                            **{f"tag.{k}": v for k, v in (s.get("tags") or {}).items()}})
        r["from"], r["to"] = min(r["from"], e["tStart"]), max(r["to"], e["tEnd"])
        r["samples"] += e["count"]
    df = pd.DataFrame(rows.values())
    for c in ("from", "to"):
        df[c] = pd.to_datetime(df[c], unit="us", utc=True)
    return df.sort_values(["metric", "series_id"]).reset_index(drop=True)


inv = client.inventory() if WAY == "A" else inventory_from_topic()
print(f"{len(inv)} series")
inv.groupby(["metric", "origin"], dropna=False).agg(series=("series_id", "count"), values=("samples", "sum"),
                                                    first=("from", "min"), last=("to", "max"))

## 2. What is measured and what is reported

The series of the year 2025: the weather of the DWD station Greifswalder Oie (measured) and the day-ahead price
(reported). The series of the turbines come from the simulator and are therefore called simulated.

In [ ]:
cols = ["metric", "unit", "from", "to", "samples", "missing", "origin"] + [c for c in inv.columns if c.startswith("tag.")]
real = inv[inv["origin"].isin(["MEASURED", "REPORTED"])]
real[[c for c in cols if c in real.columns]].dropna(axis=1, how="all")

## 3. One series as a curve

The chosen series is the 10-minute wind of the station. For another series change `METRIC` and `TAGS`, for example
to `weather.air_temperature` or to `market.price.day_ahead` with `{"area": "DE-LU"}`.

In [ ]:
METRIC = "weather.wind_speed_10min"
TAGS = {"station": "01759"}
START, END = "2025-01-01T00:00:00Z", "2026-01-01T00:00:00Z"

if WAY == "A":
    frame = client.frame(metric=METRIC, tags=TAGS, start=START, end=END)
else:
    frame = episodes_to_frame(reader.episodes(TOPIC, metric=METRIC, tags=TAGS, start=START, end=END, max_records=MAX_RECORDS))
series = frame.set_index("ts")["value"].sort_index()
print(f"{METRIC} {TAGS}: {len(series)} values, {series.index.min()} to {series.index.max()}")

fig, ax = plt.subplots(figsize=(13, 3.5))
series.plot(ax=ax, lw=0.3, alpha=0.5, label="10 minutes")
series.resample("1D").mean().plot(ax=ax, lw=1.4, label="daily mean")
ax.set_ylabel(inv.loc[inv["metric"] == METRIC, "unit"].iloc[0] if (inv["metric"] == METRIC).any() else "")
ax.set_title(f"{METRIC} {TAGS}")
ax.legend()
plt.show()

## 4. Coverage and gaps

For each day, the share of the expected values that is present. A gap is not filled; it is listed below the chart.

In [ ]:
step = series.index.to_series().diff().mode().iloc[0]
expected_per_day = pd.Timedelta("1D") / step
per_day = series.resample("1D").count() / expected_per_day

grid = per_day.to_frame("share")
grid["month"] = grid.index.month
grid["day"] = grid.index.day
heat = grid.pivot(index="month", columns="day", values="share")
fig, ax = plt.subplots(figsize=(13, 3.5))
im = ax.imshow(heat, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xlabel("Day")
ax.set_ylabel("Month")
ax.set_xticks(range(0, 31, 5), [str(d) for d in range(1, 32, 5)])
ax.set_yticks(range(len(heat.index)), [str(m) for m in heat.index])
fig.colorbar(im, ax=ax, label="Share of the expected values")
ax.set_title(f"Coverage per day, interval {step}")
plt.show()

full = pd.date_range(series.index.min(), series.index.max(), freq=step)
missing = full.difference(series.dropna().index)
runs = []
if len(missing):
    start = prev = missing[0]
    for t in missing[1:]:
        if t - prev != step:
            runs.append((start, prev + step))
            start = t
        prev = t
    runs.append((start, prev + step))
gaps = pd.DataFrame(runs, columns=["from", "to"])
gaps["duration"] = gaps["to"] - gaps["from"]
print(f"{len(missing)} missing values in {len(gaps)} gaps")
gaps.sort_values("duration", ascending=False).head(10)

## 5. The electricity price of 2025

The day-ahead price is hourly up to the delivery day 2025-09-30 and quarter-hourly after it. An hourly value does
not become a quarter-hour measurement when it is repeated; the curve shows both parts at their own resolution.

In [ ]:
PRICE = {"metric": "market.price.day_ahead", "tags": {"area": "DE-LU"}}
if WAY == "A":
    price = client.frame(**PRICE, start=START, end=END)
else:
    price = episodes_to_frame(reader.episodes(TOPIC, **PRICE, start=START, end=END, max_records=MAX_RECORDS))
price = price.set_index("ts")["value"].sort_index()
switch = pd.Timestamp("2025-09-30T22:00:00Z")
print("Resolution before the switch:", price[:switch].index.to_series().diff().mode().iloc[0],
      "| after it:", price[switch:].index.to_series().diff().mode().iloc[0])
print(f"negative prices: {(price < 0).sum()} intervals, lowest price {price.min():.2f} EUR/MWh")

fig, ax = plt.subplots(figsize=(13, 3.5))
price.plot(ax=ax, lw=0.4)
ax.axvline(switch, color="grey", ls="--", lw=1)
ax.text(switch, ax.get_ylim()[1] * 0.9, "  quarter-hourly from here", color="grey")
ax.axhline(0, color="black", lw=0.6)
ax.set_ylabel("EUR/MWh")
ax.set_title("Day-ahead price DE-LU 2025 (SMARD, reported)")
plt.show()

## Where to go next

* Notebook 2 puts a signal of the hydrogen chain next to its expectation (`client.signals("h2-chain")`).
* Notebook 3 checks a file of your own against a signal.
* Notebook 4 computes the twin (`client.scenario(...)`, `client.run(...)`).

Sources: Deutscher Wetterdienst, CC BY 4.0; Bundesnetzagentur | SMARD.de, CC BY 4.0.